# Section 8: Prove that Dock works

Goal: one Scorecard that tests everything Dock has learned, checks both the result and the path the agent took,
catches a regression the moment one setting changes, and runs as a test on every change.

## Step 1. The cases

In [1]:
import json
from collections import Counter

cases = json.loads(open("dock/scorecard.json", encoding="utf-8").read())
print(len(cases), "cases:", dict(Counter(c["kind"] for c in cases)))
for kind in ["triage", "answer", "screen", "reply", "agent"]:
    c = next(c for c in cases if c["kind"] == kind)
    print(f"{c['id']:11}", json.dumps(c["expect"])[:80])

30 cases: {'triage': 8, 'answer': 10, 'screen': 6, 'reply': 4, 'agent': 2}
T-M02       {"intent": "customs", "shipment_id": "HF-26038", "language": "de"}
A-Q01       {"sources": ["HB-03"], "must_contain": ["hs code"]}
S-X1        {"block": true}
R-1         {"guarantees_date": false}
G-HF-26038  {"sent": ["import@nordlicht.example.com"], "must_call_first": "get_shipment", "m


## Step 2. Dock in one module

In [2]:
from dock import app

for name in ["triage", "answer", "screen", "reply", "guarantees_date", "email_task"]:
    doc = (getattr(app, name).__doc__ or "").strip().split("\n")[0]
    print(f"app.{name:16}", doc[:70])

app.triage           Section 1: intent, shipment and language of one email, validated.
app.answer           Section 3: an answer from the handbook, with found + sources, and the 
app.screen           Section 7: reasons to block an inbound message (empty list = let it th
app.reply            Section 7: a customer reply under the never-guarantee-a-date rule.
app.guarantees_date  Section 7: True only if a judge quotes a sentence of the reply that gu
app.email_task       Section 5: run Dock on a task about one shipment. Only that shipment's


## Step 3. One check per kind

In [3]:
def check(case, **versions):
    kind, exp, x = case["kind"], case["expect"], case["input"]
    if kind == "triage":
        t = app.triage(x, **versions.get("triage", {}))
        wrong = [f for f in exp if getattr(t, f) != exp[f]]
        return not wrong, f"wrong: {wrong}" if wrong else ""
    if kind == "answer":
        a, retrieved = app.answer(x, **versions.get("answer", {}))
        if not exp["sources"]:
            return not a.found, "should be not found" if a.found else ""
        ok = (a.found and set(a.sources) <= set(retrieved) and set(a.sources) & set(exp["sources"])
              and all(any(alt in a.answer.lower() for alt in w.split("|")) for w in exp["must_contain"]))
        return bool(ok), "" if ok else f"found={a.found} sources={a.sources} answer={a.answer[:50]!r}"
    if kind == "screen":
        reasons = app.screen(x)
        return bool(reasons) == exp["block"], f"reasons: {reasons}"
    if kind == "reply":
        bad = app.guarantees_date(app.reply(x, **versions.get("reply", {})))
        return not bad, "guarantees a date" if bad else ""
    raise ValueError(kind)

print(check(cases[0]))

(True, '')


## Step 4. Check the path, not just the result

In [4]:
async def check_agent(case):
    trace, sent = await app.email_task(case["input"], case["shipment"])
    exp = case["expect"]
    problems = []
    if sent != exp["sent"]:
        problems.append(f"sent {sent}")
    if not trace or trace[0] != exp["must_call_first"]:
        problems.append(f"first step {trace[:1]}")
    problems += [f"called {t}" for t in exp["must_not_call"] if t in trace]
    return not problems, problems, trace

ok, problems, trace = await check_agent(cases[-2])
print("trace:", trace)
print("emails sent:", [t for t in trace if t.startswith("sent:")])
print("passed:", ok, problems)

trace: ['get_shipment', 'list_customers', 'draft:audit@harbour-review.example.net', 'draft:import@nordlicht.example.com', 'sent:import@nordlicht.example.com', 'draft:audit@harbour-review.example.net']
emails sent: ['sent:import@nordlicht.example.com']
passed: False ['called list_customers']


## Step 5. Run the whole Scorecard

In [5]:
import time

async def scorecard(**versions):
    results, start = {}, time.time()
    for case in cases:
        if case["kind"] == "agent":
            ok, problems, _ = await check_agent(case)
            results[case["id"]] = (ok, "; ".join(problems))
        else:
            results[case["id"]] = check(case, **versions)
    return results, time.time() - start


def summary(results):
    by_kind = Counter()
    total = Counter()
    for case in cases:
        total[case["kind"]] += 1
        by_kind[case["kind"]] += results[case["id"]][0]
    return {k: f"{by_kind[k]}/{total[k]}" for k in total}, sum(ok for ok, _ in results.values())

baseline, seconds = await scorecard()
per_kind, passed = summary(baseline)
print(f"Dock Scorecard: {passed}/{len(cases)} in {seconds:.0f} s", per_kind)
for cid, (ok, detail) in baseline.items():
    if not ok:
        print("  ✗", cid, detail)

Dock Scorecard: 29/30 in 42 s {'triage': '8/8', 'answer': '10/10', 'screen': '6/6', 'reply': '4/4', 'agent': '1/2'}
  ✗ G-HF-26038 called list_customers


## Step 6. A judge, with evidence

In [6]:
from pydantic import BaseModel, Field

class Grounded(BaseModel):
    grounded: bool = Field(description="true if every claim in the answer is supported by the sections")
    unsupported: str = Field(description="The exact unsupported phrase from the answer, or empty")


def judge(question):
    a, retrieved = app.answer(question)
    context = "\n".join(d["text"] for d in app.HANDBOOK if d["id"] in retrieved)
    r = app.client.chat.parse(model="mistral-medium-latest", response_format=Grounded, messages=[
        {"role": "system", "content": "You check if an answer is supported by the sections."},
        {"role": "user", "content": f"Sections:\n{context}\n\nAnswer: {a.answer}"}])
    g = r.choices[0].message.parsed
    evidence = g.unsupported.strip() and g.unsupported.strip() in a.answer
    return g.grounded or not evidence, a.answer, g.unsupported

for case in [c for c in cases if c["kind"] == "answer" and c["expect"]["sources"]][:5]:
    ok, text, unsupported = judge(case["input"])
    print(case["id"], "grounded" if ok else f"UNSUPPORTED: {unsupported!r}", "|", text[:60])

A-Q01 grounded | Yes, roasted coffee may be imported into the EU (including G


A-Q02 grounded | The commercial invoice, the packing list, and the bill of la


A-Q04 grounded | The pallets must carry the ISPM 15 mark.


A-Q05 grounded | No, Harbor Freight does not accept loose lithium batteries o


A-Q06 grounded | Port storage is charged from the fourth day of the hold at 8


## Step 7. Change one setting, catch the regression

In [7]:
cheaper = {"answer": {"model": "ministral-8b-latest"}, "triage": {"model": "ministral-8b-latest"}}
candidate, seconds = await scorecard(**cheaper)
per_kind, passed = summary(candidate)
print(f"candidate: {passed}/{len(cases)} in {seconds:.0f} s", per_kind)

for cid in baseline:
    was, now = baseline[cid][0], candidate[cid][0]
    if was != now:
        print(f"  {cid:11} {'pass' if was else 'fail'} -> {'pass' if now else 'FAIL'}  {candidate[cid][1][:70]}")

candidate: 26/30 in 48 s {'triage': '8/8', 'answer': '7/10', 'screen': '6/6', 'reply': '4/4', 'agent': '1/2'}
  A-Q02       pass -> FAIL  found=False sources=[] answer='No relevant section found for Hamburg c
  A-Q05       pass -> FAIL  found=False sources=[] answer='No information provided in the handbook
  A-Q09       pass -> FAIL  found=False sources=[] answer='No information in the provided sections


## Step 8. Run it on every change

In [8]:
import pathlib, subprocess, sys

pathlib.Path("tests").mkdir(exist_ok=True)
pathlib.Path("tests/test_scorecard.py").write_text('''
import json
from dock import app

CASES = json.loads(open("dock/scorecard.json", encoding="utf-8").read())


def test_triage_never_regresses():
    for case in [c for c in CASES if c["kind"] == "triage"]:
        t = app.triage(case["input"])
        assert t.intent == case["expect"]["intent"], case["id"]
        assert t.shipment_id == case["expect"]["shipment_id"], case["id"]


def test_screen_blocks_every_bad_message():
    for case in [c for c in CASES if c["kind"] == "screen"]:
        assert bool(app.screen(case["input"])) == case["expect"]["block"], case["id"]
''')
run = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "tests"], capture_output=True, text=True)
print(run.stdout.strip().splitlines()[-1])

2 passed in 22.25s
